In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import re
import os
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import pickle

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [2]:
# ==================== 1. Vocabulary and Text Processing ====================
class PersianVocabulary:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {'<PAD>': 0, '<UNK>': 1, '<SOS>': 2, '<EOS>': 3}
        self.idx2word = {0: '<PAD>', 1: '<UNK>', 2: '<SOS>', 3: '<EOS>'}
        self.word_freq = Counter()
        
    def build_vocabulary(self, texts):
        """Build vocabulary from list of texts"""
        for text in texts:
            words = self.tokenize_persian(text)
            self.word_freq.update(words)
        
        # Add words that meet minimum frequency
        for word, freq in self.word_freq.items():
            if freq >= self.min_freq:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word
    
    def tokenize_persian(self, text):
        """Tokenize Persian text into words"""
        if not isinstance(text, str):
            return []
        
        # Clean text - keep Persian characters, spaces, and common punctuation
        text = re.sub(r'[^\u0600-\u06FF\s\.\,\!\?]', '', text)
        text = re.sub(r'\s+', ' ', text)
        
        # Split by spaces
        words = text.strip().split()
        
        return words
    
    def numericalize(self, text, max_length=100):
        """Convert text to sequence of indices"""
        words = self.tokenize_persian(text)
        
        # Add SOS and EOS tokens
        words = ['<SOS>'] + words[:max_length-2] + ['<EOS>']
        
        # Convert to indices
        indices = [self.word2idx.get(word, self.word2idx['<UNK>']) for word in words]
        
        # Pad or truncate
        if len(indices) < max_length:
            indices += [self.word2idx['<PAD>']] * (max_length - len(indices))
        else:
            indices = indices[:max_length]
            indices[-1] = self.word2idx['<EOS>']
        
        return indices
    
    def __len__(self):
        return len(self.word2idx)

# ==================== 2. Dataset Class ====================
class PersianSentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_length=100):
        self.texts = texts
        self.labels = labels
        self.vocab = vocab
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Convert text to indices
        indices = self.vocab.numericalize(text, self.max_length)
        
        return {
            'input_ids': torch.tensor(indices, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.long)
        }

# ==================== 3. Model Definition ====================
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=256, 
                 output_dim=3, n_layers=2, dropout=0.3):
        super(SentimentClassifier, self).__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Bi-directional LSTM
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        
        self.dropout = nn.Dropout(dropout)
        
        # Attention mechanism
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )
        
        # Fully connected layers
        self.fc1 = nn.Linear(hidden_dim * 2, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
        self.relu = nn.ReLU()
        
    def forward(self, text):
        # text shape: [batch_size, seq_length]
        
        embedded = self.embedding(text)  # [batch_size, seq_length, embedding_dim]
        
        # LSTM layer
        lstm_output, (hidden, cell) = self.lstm(embedded)
        
        # Attention mechanism
        attention_weights = self.attention(lstm_output)
        attention_weights = torch.softmax(attention_weights, dim=1)
        
        # Apply attention
        context_vector = torch.sum(attention_weights * lstm_output, dim=1)
        
        # Fully connected layers
        output = self.dropout(context_vector)
        output = self.fc1(output)
        output = self.relu(output)
        output = self.dropout(output)
        output = self.fc2(output)
        
        return output

# ==================== 4. Training Functions ====================
def train_epoch(model, data_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    
    for batch in data_loader:
        input_ids = batch['input_ids'].to(device)
        labels = batch['label'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids)
        loss = criterion(outputs, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        
        # Calculate accuracy
        _, predictions = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(predictions == labels).item()
        total_samples += labels.size(0)
    
    accuracy = correct_predictions / total_samples if total_samples > 0 else 0
    avg_loss = total_loss / len(data_loader)
    
    return accuracy, avg_loss

def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct_predictions = 0
    total_samples = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(input_ids)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            
            # Calculate accuracy
            _, predictions = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(predictions == labels).item()
            total_samples += labels.size(0)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    accuracy = correct_predictions / total_samples if total_samples > 0 else 0
    avg_loss = total_loss / len(data_loader)
    
    return accuracy, avg_loss, all_predictions, all_labels

def test_examples(model, vocab, max_length):
    """Test the model with example Persian comments"""
    print("\n" + "="*70)
    print("TESTING WITH EXAMPLE COMMENTS")
    print("="*70)
    
    # Example Persian comments for each sentiment
    examples = [
        # Good (1)
        ("این محصول واقعا عالی بود. کیفیت فوق العاده ای داره", 1),
        ("خیلی خوب بود توصیه میکنم", 1),
        ("از خرید این محصول بسیار راضی هستم", 1),
        ("کیفیت عالی، قیمت مناسب", 1),
        
        # Neutral (2)
        ("محصول متوسطی بود. نه خیلی خوب نه خیلی بد", 2),
        ("نمی تونم بگم خوب بود یا بد", 2),
        ("نظر خاصی ندارم، معمولی بود", 2),
        ("هم خوب بود هم بد", 2),
        
        # Bad (3)
        ("بدترین خرابیم بود. پولم رو هدر دادم", 3),
        ("از خرید این محصول پشیمونم، اصلا خوب نبود", 3),
        ("کیفیت بسیار پایینی داشت", 3),
        ("ناراضی هستم، محصول بدی بود", 3),
    ]
    
    # Create inference model
    class InferenceWrapper:
        def __init__(self, model, vocab, max_length, device):
            self.model = model
            self.vocab = vocab
            self.max_length = max_length
            self.device = device
            self.model.eval()
            
        def predict(self, text):
            indices = self.vocab.numericalize(text, self.max_length)
            input_tensor = torch.tensor(indices, dtype=torch.long).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outputs = self.model(input_tensor)
                probabilities = torch.softmax(outputs, dim=1)
                _, prediction = torch.max(outputs, dim=1)
            
            # Map back to 1,2,3
            suggestion = prediction.item() + 1
            probs = probabilities[0].cpu().numpy()
            
            return suggestion, probs
    
    inference_model = InferenceWrapper(model, vocab, max_length, device)
    
    correct = 0
    total = len(examples)
    
    for i, (text, expected) in enumerate(examples, 1):
        predicted, probs = inference_model.predict(text)
        
        status = "✓" if predicted == expected else "✗"
        if predicted == expected:
            correct += 1
        
        print(f"\nExample {i}:")
        print(f"Text: {text}")
        print(f"Expected: {expected} ({'Good' if expected==1 else 'Neutral' if expected==2 else 'Bad'})")
        print(f"Predicted: {predicted} ({'Good' if predicted==1 else 'Neutral' if predicted==2 else 'Bad'}) {status}")
        print(f"Probabilities: Good: {probs[0]:.2%}, Neutral: {probs[1]:.2%}, Bad: {probs[2]:.2%}")
        print("-" * 60)
    
    print(f"\nTest Accuracy: {correct}/{total} = {correct/total:.2%}")

In [3]:
print("Loading dataset...")
df = pd.read_csv('digikala_sentiment.csv')

# Check data
print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# Map suggestions to labels (keeping original values 1,2,3)
# 1 = Good (Positive), 2 = Neutral, 3 = Bad (Negative)
df['label'] = df['Suggestion'].copy()  # Keep original values 1,2,3

# Convert to 0-indexed for PyTorch
df['label_idx'] = df['label'] - 1  # Convert 1,2,3 to 0,1,2 for training

print(f"\nSuggestion distribution:")
print(df['label'].value_counts().sort_index())
print("\nMapping:")
print("1 → Good (Positive)")
print("2 → Neutral")
print("3 → Bad (Negative)")

# Split data
X = df['Text'].astype(str).values
y = df['label_idx'].values  # Use 0-indexed labels for training
y_original = df['label'].values  # Keep original 1,2,3 values

# Create train/val/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"\nData split:")
print(f"Train: {len(X_train)} samples")
print(f"Validation: {len(X_val)} samples")
print(f"Test: {len(X_test)} samples")

# Build vocabulary from training data
print("\nBuilding vocabulary...")
vocab = PersianVocabulary(min_freq=2)
vocab.build_vocabulary(X_train)
print(f"Vocabulary size: {len(vocab)}")
print(f"Most common words: {vocab.word_freq.most_common(10)}")

# Create datasets
max_length = 100
batch_size = 32

train_dataset = PersianSentimentDataset(X_train, y_train, vocab, max_length)
val_dataset = PersianSentimentDataset(X_val, y_val, vocab, max_length)
test_dataset = PersianSentimentDataset(X_test, y_test, vocab, max_length)

# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# Initialize model
print("\nInitializing model...")
model = SentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    hidden_dim=128,
    output_dim=3,  # 3 classes: Good, Neutral, Bad
    n_layers=2,
    dropout=0.1
)

model = model.to(device)

# Training setup
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

# Training loop
num_epochs = 10
best_val_accuracy = 0
patience = 5
patience_counter = 0

print(f"\nStarting training for {num_epochs} epochs...")
print("-" * 60)

training_history = []

for epoch in range(num_epochs):
    # Training
    train_accuracy, train_loss = train_epoch(
        model, train_loader, criterion, optimizer, device
    )
    
    # Validation
    val_accuracy, val_loss, _, _ = evaluate(
        model, val_loader, criterion, device
    )
    
    # Store history
    training_history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss,
        'train_acc': train_accuracy,
        'val_loss': val_loss,
        'val_acc': val_accuracy
    })
    
    # Print progress
    print(f"Epoch {epoch + 1:02}/{num_epochs}")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy:.4f}")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_accuracy:.4f}")
    
    # Learning rate scheduling
    scheduler.step(val_accuracy)
    
    # Save best model
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        patience_counter = 0
        
        # Save model
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'vocab': vocab,
            'max_length': max_length,
            'val_accuracy': val_accuracy,
            'training_history': training_history
        }, 'best_sentiment_model.pth')
        
        print(f"  ✓ Model saved (val_acc: {val_accuracy:.4f})")
    else:
        patience_counter += 1
        
    # Early stopping
    if patience_counter >= patience:
        print(f"\nEarly stopping triggered after {epoch + 1} epochs")
        break
    
    print("-" * 60)

Loading dataset...
Dataset shape: (3261, 3)
Columns: ['Text', 'Score', 'Suggestion']

Suggestion distribution:
label
1    2382
2     419
3     460
Name: count, dtype: int64

Mapping:
1 → Good (Positive)
2 → Neutral
3 → Bad (Negative)

Data split:
Train: 2282 samples
Validation: 489 samples
Test: 490 samples

Building vocabulary...
Vocabulary size: 4596
Most common words: [('و', 4879), ('که', 2585), ('از', 2476), ('این', 2136), ('به', 2088), ('با', 1583), ('رو', 1571), ('در', 1370), ('هم', 1340), ('من', 1308)]

Initializing model...

Starting training for 10 epochs...
------------------------------------------------------------
Epoch 01/10
  Train Loss: 0.7979 | Train Acc: 0.7152
  Val Loss:   0.7777 | Val Acc:   0.7301
  ✓ Model saved (val_acc: 0.7301)
------------------------------------------------------------
Epoch 02/10
  Train Loss: 0.7409 | Train Acc: 0.7336
  Val Loss:   0.7259 | Val Acc:   0.7342
  ✓ Model saved (val_acc: 0.7342)
------------------------------------------------

In [12]:
print("\nLoading best model for testing...")
checkpoint = torch.load('best_sentiment_model.pth', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Test the model
test_accuracy, test_loss, test_predictions, test_labels = evaluate(
    model, test_loader, criterion, device
)

# Convert predictions back to original labels (1,2,3)
test_predictions_original = [p + 1 for p in test_predictions]
test_labels_original = [l + 1 for l in test_labels]

print(f"\nTest Results:")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Confusion matrix
print(f"\nConfusion Matrix:")
cm = confusion_matrix(test_labels_original, test_predictions_original, labels=[1, 2, 3])
cm_df = pd.DataFrame(cm, 
                     index=['Actual Good(1)', 'Actual Neutral(2)', 'Actual Bad(3)'],
                     columns=['Pred Good(1)', 'Pred Neutral(2)', 'Pred Bad(3)'])
print(cm_df)

print(f"\nClassification Report:")
print(classification_report(
    test_labels_original, 
    test_predictions_original,
    target_names=['Good (1)', 'Neutral (2)', 'Bad (3)']
))


Loading best model for testing...

Test Results:
Test Loss: 0.7503
Test Accuracy: 0.7306

Confusion Matrix:
                   Pred Good(1)  Pred Neutral(2)  Pred Bad(3)
Actual Good(1)              358                0            0
Actual Neutral(2)            63                0            0
Actual Bad(3)                69                0            0

Classification Report:
              precision    recall  f1-score   support

    Good (1)       0.73      1.00      0.84       358
 Neutral (2)       0.00      0.00      0.00        63
     Bad (3)       0.00      0.00      0.00        69

    accuracy                           0.73       490
   macro avg       0.24      0.33      0.28       490
weighted avg       0.53      0.73      0.62       490



C:\Users\Masoud\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Masoud\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Masoud\AppData\Roaming\Python\Python313\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [17]:
# Modify the save section in your training script:
def save_model_with_vocab(model, vocab, optimizer, epoch, val_accuracy, max_length=100):
    """Save model with vocabulary"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'vocab_word2idx': vocab.word2idx,  # Save the dictionary
        'vocab_size': len(vocab),
        'max_length': max_length,
        'val_accuracy': val_accuracy,
        'training_history': training_history
    }
    
    torch.save(checkpoint, 'best_sentiment_model.pth')
    print(f"Model saved with vocabulary to best_sentiment_model.pth")
    
    # Also save a separate vocabulary file for compatibility
    import json
    with open('vocab_dict.json', 'w', encoding='utf-8') as f:
        json.dump(vocab.word2idx, f, ensure_ascii=False)
    print("Vocabulary saved to vocab_dict.json")

# In your main() function, replace the saving part with:
save_model_with_vocab(model, vocab, optimizer, epoch, val_accuracy, max_length)

Model saved with vocabulary to best_sentiment_model.pth
Vocabulary saved to vocab_dict.json


In [13]:
# MasoudKaviani.ir